# 08 — PCA dhe reduktimi i dimensionalitetit

        **Pyetja udhëheqëse:** Si gjejmë drejtimet ku të dhënat fizike ndryshojnë më shumë?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të ndërtohet matrica e kovariancës.
- Të nxirren vlerat/vetorët vetjakë.
- Të kuptohet varianca e shpjeguar dhe rikonstruksioni.
- Të implementohet PCA manualisht.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

## Pse kjo temë është e nevojshme?

Në eksperimentet moderne shpesh masim shumë madhësi të korreluara. PCA pyet nëse pjesa më e madhe e variacionit mund të përshkruhet me pak drejtime të reja, të ndërtuara nga vetë të dhënat. Kjo është një pyetje gjeometrike, statistikore dhe numerike njëkohësisht.

Duam të shohim drejtpërdrejt se si kovarianca çon te vlerat dhe vektorët vetjakë, si krijohen komponentët kryesorë dhe çfarë humbim kur reduktojmë dimensionin. Komponentët nuk duhen identifikuar automatikisht me shkaqe fizike.

### Si duhet lexuar ky notebook

Për çdo bllok kodi do të shpjegojmë **pse** kryhet ai hap, çfarë pyetjeje i përgjigjet dhe çfarë duhet të kontrollojmë në dalje. Kodi nuk trajtohet si recetë: çdo grafik dhe çdo metrikë duhet të rikthehet te hipoteza matematikore dhe te interpretimi fizik. Kur një rezultat nuk sillet siç presim, pyetja e parë nuk është “si ta rrisim score-in?”, por “cilën hipotezë të modelit ose të të dhënave po shkelim?”.

### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("08 — PCA dhe reduktimi i dimensionalitetit")
print("Seed:", RANDOM_STATE)

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 1. Spektra të korreluara — CORE

### Para se të ekzekutojmë: Spektra me pak faktorë latentë

Ndërtojmë spektra me shumë gjatësi valore, por me vetëm dy amplituda kryesore që ndryshojnë. Pra dimensioni i matur është i madh, ndërsa dimensioni efektiv i strukturës është i vogël. Ky është skenari ideal për të kuptuar PCA.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Spektra me pak faktorë latentë.
wl=np.linspace(400,750,120)
n=260
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
a=rng.normal(1.0,0.12,n)
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
b=rng.normal(0.5,0.09,n)
spectra=[]
for ai,bi in zip(a,b):
    s=ai*np.exp(-0.5*((wl-520)/28)**2)+bi*np.exp(-0.5*((wl-640)/35)**2)
    # Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
    s+=rng.normal(0,0.02,len(wl))
    spectra.append(s)
X=np.array(spectra)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
# Përsërisim procedurën për të studiuar konvergjencën ose ndjeshmërinë ndaj parametrave.
for i in range(20):
    ax.plot(wl,X[i],alpha=0.5)
ax.set_xlabel("Gjatësia valore [nm]"); ax.set_ylabel("Intensitet")
ax.set_title("Shembuj spektrash")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 2. Nga kovarianca te drejtimet kryesore — CORE

Pas qendrimit $X_c=X-\bar X$, matrica e kovariancës është
$$
C=\frac{1}{N-1}X_c^TX_c.
$$
Drejtimet kryesore plotësojnë
$$
C\mathbf v_i=\lambda_i\mathbf v_i.
$$
Vlera vetjake $\lambda_i$ mat variancën përgjatë komponentit përkatës.

### Para se të ekzekutojmë: Ndërtimi manual i PCA

Qendrojmë të dhënat, ndërtojmë kovariancën dhe diagonalizojmë matricën. Renditja e eigenvalues nga më e madhja te më e vogla identifikon drejtimet e variacionit maksimal.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Ndërtimi manual i PCA.
Xc=X-X.mean(axis=0)
# Kovarianca përmbledh bashkë-ndryshimin e veçorive pas qendrimit të të dhënave.
C=np.cov(Xc,rowvar=False)
# Diagonalizojmë matricën simetrike të kovariancës për të marrë komponentët kryesorë.
vals,vecs=np.linalg.eigh(C)
# Renditja përdoret për të identifikuar fqinjët ose komponentët sipas kriterit përkatës.
idx=np.argsort(vals)[::-1]
vals,vecs=vals[idx],vecs[:,idx]
explained=vals/vals.sum()

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.semilogy(np.arange(1,21),vals[:20],marker="o")
ax.set_xlabel("Komponenti"); ax.set_ylabel("Vlera vetjake")
ax.set_title("Spektri i vlerave vetjake")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

### PCA nga paketa didaktike `src/`

### Para se të ekzekutojmë: Krahasim me modulin kanonik

Përdorim `PCAFromScratch` për të verifikuar se procedura e paketizuar jep të njëjtën variancë të shpjeguar. Kjo është një kontroll i drejtpërdrejtë i implementimit manual.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Krahasim me modulin kanonik.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import sys
# Kërkojmë automatikisht rrënjën e repository-t, në mënyrë që notebook-u të funksionojë nga dosje të ndryshme.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from src.pca.pca_from_scratch import PCAFromScratch
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
pca_src = PCAFromScratch(n_components=2).fit(X)
print("Varianca e shpjeguar (src):", np.round(pca_src.explained_variance_ratio_, 5))

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 3. Varianca kumulative — CORE

### Para se të ekzekutojmë: Sa komponentë ruajmë?

Varianca kumulative e shpjeguar është një mënyrë praktike për të vendosur sa dimensione të ruajmë. Pragu 90% ose 95% është konvencion, jo ligj fizik; zgjedhja duhet lidhur me qëllimin e analizës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Sa komponentë ruajmë?.
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(np.cumsum(explained[:20]),marker="o")
ax.set_xlabel("Numri i komponentëve")
ax.set_ylabel("Varianca kumulative e shpjeguar")
ax.set_ylim(0,1.02)
ax.set_title("Sa dimensione duhen?")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 4. Projekcioni në hapësirën PCA — CORE

### Para se të ekzekutojmë: Projekcioni në koordinata të reja

Koordinatat PCA janë projektime të spektrave mbi drejtimet kryesore. Ngjyrosja me një parametër latent na ndihmon të shohim nëse komponenti ka lidhje me një variacion fizik të njohur.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Projekcioni në koordinata të reja.
Z=Xc@vecs[:,:2]
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
scat=ax.scatter(Z[:,0],Z[:,1],c=a,s=16)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Projekcioni PCA; ngjyra kodon amplitudën e pikut të parë")
plt.colorbar(scat,ax=ax,label="a")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 5. Loadings dhe kuptimi fizik — CORE

### Para se të ekzekutojmë: Loadings dhe interpretimi spektral

Loadings tregojnë cilat gjatësi valore kontribuojnë te një komponent. Struktura e tyre mund të ketë kuptim fizik, por komponenti është përcaktuar nga varianca, jo nga një model shkakësor.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Loadings dhe interpretimi spektral.
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(wl,vecs[:,0],label="PC1 loading")
ax.plot(wl,vecs[:,1],label="PC2 loading")
ax.set_xlabel("Gjatësia valore [nm]")
ax.set_ylabel("Koeficienti")
ax.legend(); ax.set_title("Çfarë strukturash spektrale kap PCA?")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 6. Rikonstruksioni — CORE

### Para se të ekzekutojmë: Rikonstruksioni dhe humbja e informacionit

Rikonstruktojmë të njëjtin spektër me numër në rritje komponentësh. Kjo e bën të dukshme çfarë informacioni ruhet dhe çfarë hidhet poshtë gjatë reduktimit dimensional.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Rikonstruksioni dhe humbja e informacionit.
sample=7
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(wl,X[sample],label="origjinali")
for k in [1,2,5,15]:
    recon=X.mean(axis=0)+(Xc[sample]@vecs[:,:k])@vecs[:,:k].T
    ax.plot(wl,recon,label=f"k={k}")
ax.legend(); ax.set_xlabel("nm"); ax.set_ylabel("Intensitet")
ax.set_title("Rikonstruksioni me numër të ndryshëm komponentësh")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 7. Krahasim me sklearn — EXTENDED

### Para se të ekzekutojmë: Krahasimi me implementimin profesional

Në fund verifikojmë raportet e variancës me `sklearn`. Shenja e eigenvectors mund të ndryshojë, por nën-hapësira dhe eigenvalues duhet të përputhen.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Krahasimi me implementimin profesional.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.decomposition import PCA
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
p=PCA(n_components=5).fit(X)
print("Manual:",np.round(explained[:5],5))
print("sklearn:",np.round(p.explained_variance_ratio_,5))

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 8. Aplikim jo-fizik: të dhëna biometrike — EXTENDED

Në biologji ose mjekësi, qindra matje të korreluara mund të reduktohen në pak drejtime dominante. Kjo lehtëson vizualizimin, por komponentët nuk janë automatikisht variabla kauzale.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Verifikoni ortogonalitetin e komponentëve të parë.
2. Llogaritni gabimin e rikonstruksionit ndaj k.
3. Standardizoni çdo gjatësi valore dhe përsëritni PCA.
4. Shpjegoni pse shenja e një eigenvector-i është arbitrare.

        ## B. Ushtrim i plotë
        Shtoni një pik spektral të tretë me amplitudë të pavarur dhe kontrolloni sa komponentë duhen për 95% variancë.

        ## C. Playground
        Ndryshoni numrin e mostrave, zhurmën, amplitudat dhe numrin e komponentëve.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **PCA dhe reduktimi i dimensionalitetit** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.